In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from statsmodels.tsa.stattools import coint
from itertools import combinations

# Atsisiunciame duomenis ( pvz . NVDA ir BTC )
tickers = ["BTC-USD", "DX-Y.NYB", "GC=F", "BZ=F"]
df = yf.download(tickers, start="2014-01-01")

# Suvienodinti dazni, nes portfeli sudaro kripto ir derivatyvai
df = df.dropna()

# Apskaiciuojame grazas
returns = df["Close"].pct_change().dropna()


[*********************100%***********************]  4 of 4 completed


In [2]:
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)


In [3]:
# 1 užduotis
def compute_rsi(price_series: pd.Series, window: int = 14) -> pd.Series:
    delta = price_series.diff()

    gains = delta.clip(lower=0)
    losses = -delta.clip(upper=0)

    avg_gain = gains.rolling(window=window, min_periods=window).mean()
    avg_loss = losses.rolling(window=window, min_periods=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_golden_cross_features(
    asset_df: pd.DataFrame, short_window: int = 50, long_window: int = 200
) -> pd.DataFrame:

    asset_df["ma50"] = (
        asset_df["Close"].rolling(window=short_window, min_periods=short_window).mean()
    )
    asset_df["ma200"] = (
        asset_df["Close"].rolling(window=long_window, min_periods=long_window).mean()
    )

    state = (asset_df["ma50"] > asset_df["ma200"]).fillna(False)
    prev_state = state.shift(1, fill_value=False)

    asset_df["in_golden_cross"] = state
    asset_df["golden_cross_event"] = state & (~prev_state)
    asset_df["death_cross_event"] = (~state) & prev_state

    return asset_df


def build_cross_features(
    asset_series: pd.Series, short_window: int = 50, long_window: int = 200
) -> pd.DataFrame:
    asset_df = pd.DataFrame({"Close": asset_series}).copy()

    asset_df = compute_golden_cross_features(asset_df, short_window, long_window)

    asset_df["rsi14"] = compute_rsi(asset_df["Close"], window=14)

    return asset_df


def analyze_assets(df: pd.DataFrame, weights: list[float]) -> dict[str, pd.DataFrame]:
    close_prices = df["Close"]

    assert np.isclose(sum(weights), 1), "Svoriai turi susideti i 1"
    assert (
        len(weights) == close_prices.shape[1]
    ), "Svoriu skaicius turi buti toks pats kaip ir aktyvu"

    assets_data: dict[str, pd.DataFrame] = {}
    for asset in close_prices.columns:
        asset_series = close_prices[asset]
        assets_data[asset] = build_cross_features(asset_series)

    return assets_data


In [ ]:
assets_data = analyze_assets(df, [0.5, 0.2, 0.1, 0.2])

btc = assets_data["BTC-USD"].copy()

btc[-20:]

In [6]:
# read data\BTC_news_articles.csv
btc_news_df = pd.read_csv("data/BTC_news_articles.csv", sep=';')
btc_news_df = btc_news_df[["title", "newsDatetime"]]
print(btc_news_df)

                                                  title      newsDatetime
0     Bitcoin (BTC) Loses 200-Day MA, Tries to Hold ...  2021-08-18 09:50
1     Crypto Analyst Lark Davis on Bitcoin: ‘Still G...  2021-08-18 10:10
2                             Where I am BUYING Bitcoin  2021-08-18 18:27
3     Twitter’s Jack Dorsey Is Now Mining Bitcoin. H...  2021-08-18 18:56
4     Bitcoin Trading Volume Sinks Without Decisive ...  2021-08-18 19:00
...                                                 ...               ...
4868  Despite Identical Monetary Tailwinds, Gold is ...  2025-12-03 10:00
4869  $93K And Climbing: Analysts Say Bitcoin’s Push...  2025-12-03 10:00
4870  Major US Bank Recommends Wealthy Clients Consi...  2025-12-03 10:15
4871  Here’s Bitcoin’s Next Big Target After $93K Br...  2025-12-03 10:17
4872  Bitcoin Short-Term Holder Shakeout Could Accel...  2025-12-03 10:23

[4873 rows x 2 columns]


In [ ]:
# read data\gold_1.xlsx
gold_news_df_1 = pd.read_excel("data/gold_1.xlsx")
gold_news_df_2 = pd.read_excel("data/gold_2.xlsx")
gold_news_df_3 = pd.read_excel("data/gold_3.xlsx")


Index(['Title', 'Submitted Date', 'Main Content', 'Source'], dtype='str') Index(['date', 'title'], dtype='str') Index(['Title', 'Published Date', 'Main Content', 'Source'], dtype='str')
                                               Title Submitted Date
0  Comex Gold Speculators 'Miss the Move' as Bull...     2013-10-02
1  Gold Price Hits $3000 as US Consumer Sentiment...     2016-03-07
2  Gold Hits New Dollar Record, $25 Off $3000, as...     2016-10-07
3  $33 Silver Cuts Gold/Silver Ratio as US Stocks...     2016-11-09
4  Gold Rises, Silver Spikes as Stocks Slump Agai...     2016-11-10
                       date                                              title
0  2024-12-04T16:06:03-0500  Gold is never going back to $2,000: 'That's hi...
1  2024-12-02T18:31:12-0500  Gold's U.S. market share to quadruple? What ha...
2  2024-11-22T15:20:17-0500  Texas proposes gold and silver-backed currenci...
3  2024-11-21T13:47:02-0500  Solid Gold: Brooklyn Museum unveils 500 dazzli...
4  2024-11-

In [8]:
pip install openpyxl 

Note: you may need to restart the kernel to use updated packages.


In [ ]:
gold_news_df_1 = gold_news_df_1.rename(
    columns={'Title':'title', 'Submitted Date':'date'}
)

gold_news_df_2 = gold_news_df_2.rename(
    columns={'title':'title', 'date':'date'}
)

gold_news_df_3 = gold_news_df_3.rename(
    columns={'Title':'title', 'Published Date':'date'}
)

for df in (gold_news_df_1, gold_news_df_2, gold_news_df_3):
    df['date'] = pd.to_datetime(df['date'], utc=True)

In [10]:
all_gold = pd.concat([gold_news_df_1, gold_news_df_2, gold_news_df_3],
                     ignore_index=True)

all_gold = all_gold.sort_values('date')

In [11]:
print(all_gold)

                                                 title                      date                                       Main Content        Source
0    Comex Gold Speculators 'Miss the Move' as Bull... 2013-10-02 00:00:00+00:00  GOLD BULLION briefly touched $3000 again on Mo...  bullionvault
1    Gold Price Hits $3000 as US Consumer Sentiment... 2016-03-07 00:00:00+00:00  GOLD pulled back Friday after breaking $3000 p...  bullionvault
2    Gold Hits New Dollar Record, $25 Off $3000, as... 2016-10-07 00:00:00+00:00  The PRICE of GOLD jumped through last month's ...  bullionvault
3    $33 Silver Cuts Gold/Silver Ratio as US Stocks... 2016-11-09 00:00:00+00:00  GOLD held steady but silver rose yet again Wed...  bullionvault
4    Gold Rises, Silver Spikes as Stocks Slump Agai... 2016-11-10 00:00:00+00:00  GOLD PRICES rebounded and silver hit 2-week hi...  bullionvault
..                                                 ...                       ...                                            

In [ ]:
# read data\oil_sentiment_headlines.csv
oil_news_df = pd.read_csv("data/oil_sentiment_headlines.csv")[["date", "headline"]]

oil_news_df

,date,headline
0,2019-01-03,$83.7 billion in Iraq &apos; s oil revenues in...
1,2019-01-10,Study: soda destroys kidneys.
2,2019-01-14,American official: We want Qatar to challenge ...
3,2019-01-15,"Stone decor in houses... happiness, warmth and..."
4,2019-01-15,From inside Haftar prisons... shocking account...
...,...,...
11063,2026-03-10,Iran Vows To Fight On After Trump Hints At Ear...
11064,2026-03-10,"US Dollar Is Still the Dominant Currency, Tema..."
11065,2026-03-10,Trump Signals Possible End to Iran War; Oil sl...
11066,2026-03-10,Today will be 'most intense day' of strikes on...
